# Example Pipeline
Example pipeline to serve as a template

In [1]:
STEP = 'ENVIRONMENT_PREPARATION'

## 1. Environment preparation

In [2]:
# imports
import os
import sys
import json
import time
import asyncio
import polars as pl
from pathlib import Path
from playwright.async_api import TimeoutError
# import nest_asyncio
# nest_asyncio.apply()


In [3]:

EXAMPLE_INSTANCE_ID = '8ba7dff4-7b4d-4e7b-833d-b760dab603da'
DECODO_USERNAME = os.getenv("DECODO_USERNAME")
DECODO_PASSWORD = os.getenv("DECODO_PASSWORD")
DECODO_URL = os.getenv("DECODO_URL")
SELENIUM_REMOTE_URL = os.getenv("SELENIUM_REMOTE_URL")
PIPELINE_API_SECRET = os.getenv("PIPELINE_API_SECRET")

IS_DEVELOPMENT = True

PIPELINE_NAME = "aliexpress_store_pipeline"

In [4]:


# Local utilities detection
if IS_DEVELOPMENT:
    module_path = os.path.abspath(os.path.join('..'))
    if module_path not in sys.path:
        sys.path.append(module_path)
else:
    script_dir = Path(__file__).resolve().parent
    module_path = script_dir.parent
    if str(module_path) not in sys.path:
        sys.path.append(str(module_path))

from scrapers_utils import BaseScraper
from pipeline_utils.exception_handler import handle_error
from pipeline_utils import RunManager
from pipeline_utils.cleaning_utils import strip_all_str
from pipeline_utils.drive_api import DriveApi

# Parameter handling
if not IS_DEVELOPMENT:
    RUN_ID = sys.argv[1]
    PARAMS = json.loads(sys.argv[2])
    run = RunManager(RUN_ID)
    headless = True
else:
    PARAMS = {
        "listings_urls": [
            "https://es.aliexpress.com/item/1005009141784587.html?spm=a2g0o.store_pc_home.promoteWysiwyg_2010150883161.1005009141784587&gatewayAdapt=glo2esp",
            "https://es.aliexpress.com/item/1005011764425819.html?spm=a2g0o.store_pc_home.promoteWysiwyg_2010150883161.1005011764425819&gatewayAdapt=glo2esp",
            "https://es.aliexpress.com/item/1005009268797761.html?pdp_npi=4%40dis%21MXN%2182.88+MXN%24%2172.90+MXN%24%21%21%2131.59%2127.79%21%402101d2e717896572354925982e0e3c%2112000048552089535%21sh%21MX%21895102358%21X&spm=a2g0o.store_pc_allItems_or_groupList.new_all_items_2010150883223.1005009268797761&gatewayAdapt=glo2esp",
            "https://es.aliexpress.com/item/1005011942035105.html?spm=a2g0o.store_pc_allItems_or_groupList.0.0.43045a997SibTa&pdp_npi=4%40dis%21MXN%21172.21+MXN%24%21165.90+MXN%24%21%21%219.78%219.42%21%4021030cd817896190941421341e0e25%2112000057112364026%21sh%21MX%21895102358%21X&_gl=1*o13vdc*_gcl_aw*R0NMLjE3ODk2MTgzODUuQ2p3S0NBandfS2pWQmhBSEVpd0FuQzBOOUhld3BXcHB3VlRqUW1OZkFseVd3b3VyM3NJb1NhM1VURGwySGxBNjhKZGdHeUk2MDUzNE9Sb0N2b2NRQXZEX0J3RQ..*_gcl_dc*R0NMLjE3ODk2MTgzODUuQ2p3S0NBandfS2pWQmhBSEVpd0FuQzBOOUhld3BXcHB3VlRqUW1OZkFseVd3b3VyM3NJb1NhM1VURGwySGxBNjhKZGdHeUk2MDUzNE9Sb0N2b2NRQXZEX0J3RQ..*_gcl_au*NDAwMTA4MjYzLjE3ODQ2NDgxNjc.*_ga*ODAxMTU1NzM2LjE3NzY3OTgxOTU.*_ga_VED1YSGNC7*czE3ODk2MTgzNjEkbzgkZzEkdDE3ODk2MTkwOTUkajE5JGwwJGgw&gatewayAdapt=glo2esp",
            "https://es.aliexpress.com/item/1005012308067099.html?spm=a2g0o.store_pc_allItems_or_groupList.0.0.43045a997SibTa&pdp_npi=4%40dis%21MXN%211%2C246.86+MXN%24%21635.90+MXN%24%21%21%2170.80%2136.11%21%4021030cd817896187107086825e0e25%2112000058095122510%21sh%21MX%21895102358%21X&_gl=1*1aqi1s4*_gcl_aw*R0NMLjE3ODk2MTgzODUuQ2p3S0NBandfS2pWQmhBSEVpd0FuQzBOOUhld3BXcHB3VlRqUW1OZkFseVd3b3VyM3NJb1NhM1VURGwySGxBNjhKZGdHeUk2MDUzNE9Sb0N2b2NRQXZEX0J3RQ..*_gcl_dc*R0NMLjE3ODk2MTgzODUuQ2p3S0NBandfS2pWQmhBSEVpd0FuQzBOOUhld3BXcHB3VlRqUW1OZkFseVd3b3VyM3NJb1NhM1VURGwySGxBNjhKZGdHeUk2MDUzNE9Sb0N2b2NRQXZEX0J3RQ..*_gcl_au*NDAwMTA4MjYzLjE3ODQ2NDgxNjc.*_ga*ODAxMTU1NzM2LjE3NzY3OTgxOTU.*_ga_VED1YSGNC7*czE3ODk2MTgzNjEkbzgkZzEkdDE3ODk2MTg3MDgkajUkbDAkaDA.&gatewayAdapt=glo2esp",
            "https://es.aliexpress.com/item/1005011960313933.html?spm=a2g0o.productlist.main.9.21111ceeJMuhZd&algo_pvid=124d52df-ef0f-478d-9a66-d5f50571e5f8&algo_exp_id=124d52df-ef0f-478d-9a66-d5f50571e5f8-8&pdp_ext_f=%7B%22order%22%3A%22-1%22%2C%22eval%22%3A%221%22%2C%22fromPage%22%3A%22search%22%7D&pdp_npi=6%40dis%21MXN%21793.66%21763.90%21%21%2145.06%2143.37%21%4021030cd817896184177823334e0c9b%2112000057157892190%21sea%21MX%21895102358%21X%211%210%21n_tag%3A-29919%3Bd%3A9a638db6%3Bm03_new_user%3A-29895&curPageLogUid=GEKvIiQLG4dk&utparam-url=scene%3Asearch%7Cquery_from%3A%7Cx_object_id%3A1005011960313933%7C_p_origin_prod%3A",
            "https://es.aliexpress.com/item/1005009268797761.html?pdp_npi=4%40dis%21MXN%2182.88+MXN%24%2172.90+MXN%24%21%21%2131.59%2127.79%21%402101d2e717896572354925982e0e3c%2112000048552089535%21sh%21MX%21895102358%21X&spm=a2g0o.store_pc_allItems_or_groupList.new_all_items_2010150883223.1005009268797761&gatewayAdapt=glo2esp",
            "https://es.aliexpress.com/item/1005008875165003.html?spm=a2g0o.productlist.main.4.7f1618d3NkUuQZ&aem_p4p_detail=202609162126501252617311745000000135374&algo_pvid=60d84d26-eabb-4746-8c97-b7cf0f36f73b&algo_exp_id=60d84d26-eabb-4746-8c97-b7cf0f36f73b-3&pdp_ext_f=%7B%22order%22%3A%22-1%22%2C%22spu_best_type%22%3A%22price%22%2C%22eval%22%3A%221%22%2C%22fromPage%22%3A%22search%22%7D&pdp_npi=6%40dis%21MXN%2177.56%2166.14%21%21%2129.56%2125.21%21%40210311c217896192108697324e0e32%2112000052881000092%21sea%21MX%21895102358%21X%211%210%21n_tag%3A-29919%3Bd%3A9a638db6%3Bm03_new_user%3A-29895%3BpisId%3A5000000217788164&curPageLogUid=H9mYS3qqyU2d&utparam-url=scene%3Asearch%7Cquery_from%3A%7Cx_object_id%3A1005008875165003%7C_p_origin_prod%3A&search_p4p_id=202609162126501252617311745000000135374_1",
            "https://es.aliexpress.com/item/1005011914108981.html?pdp_npi=4%40dis%21MXN%2195.62+MXN%24%2145.90+MXN%24%21%21%2136.44%2117.49%21%402101e75417896585370655866e0fe3%2112000057079121322%21sh%21MX%21895102358%21X&spm=a2g0o.store_pc_allItems_or_groupList.new_all_items_2009573860043.1005011914108981&gatewayAdapt=glo2esp",
            "https://es.aliexpress.com/item/1005011878729404.html?pdp_npi=4%40dis%21MXN%21114.37+MXN%24%2154.90+MXN%24%21%21%2143.59%2120.93%21%402101e75417896585370655866e0fe3%2112000056908548249%21sh%21MX%21895102358%21X&spm=a2g0o.store_pc_allItems_or_groupList.new_all_items_2009573860043.1005011878729404&gatewayAdapt=glo2esp",
            "https://es.aliexpress.com/item/1005006547658160.html?spm=a2g0o.detail.pcDetailTopMoreOtherSeller.4.3836HMyFHMyFgI&gps-id=pcDetailTopMoreOtherSeller&scm=1007.40050.354490.0&scm_id=1007.40050.354490.0&scm-url=1007.40050.354490.0&pvid=ae6d4b9f-c9a3-45f5-99ab-6f2d7a3da9af&_t=gps-id%3ApcDetailTopMoreOtherSeller%2Cscm-url%3A1007.40050.354490.0%2Cpvid%3Aae6d4b9f-c9a3-45f5-99ab-6f2d7a3da9af%2Ctpp_buckets%3A668%232846%238107%231934&pdp_ext_f=%7B%22order%22%3A%22-1%22%2C%22eval%22%3A%221%22%2C%22sceneId%22%3A%2230050%22%2C%22fromPage%22%3A%22recommend%22%7D&pdp_npi=6%40dis%21MXN%2176.90%2176.90%21%21%2129.31%2129.31%21%402101d3fe17896592610061690e1299%2112000037629614272%21rec%21MX%21895102358%21X%211%210%21n_tag%3A-29919%3Bd%3A9a638db6%3Bm03_new_user%3A-29895&utparam-url=scene%3ApcDetailTopMoreOtherSeller%7Cquery_from%3A%7Cx_object_id%3A1005006547658160%7C_p_origin_prod%3A",
            "https://es.aliexpress.com/item/1005008797499352.html?spm=a2g0o.detail.pcDetailTopMoreOtherSeller.2.5508XIBSXIBS02&gps-id=pcDetailTopMoreOtherSeller&scm=1007.40050.354490.0&scm_id=1007.40050.354490.0&scm-url=1007.40050.354490.0&pvid=bf86fbb0-fd2b-4784-9f40-5bcb6cf78029&_t=gps-id%3ApcDetailTopMoreOtherSeller%2Cscm-url%3A1007.40050.354490.0%2Cpvid%3Abf86fbb0-fd2b-4784-9f40-5bcb6cf78029%2Ctpp_buckets%3A668%232846%238107%231934&pdp_ext_f=%7B%22order%22%3A%22-1%22%2C%22spu_best_type%22%3A%22price%22%2C%22eval%22%3A%221%22%2C%22sceneId%22%3A%2230050%22%2C%22fromPage%22%3A%22recommend%22%7D&pdp_npi=6%40dis%21MXN%21179.78%2180.90%21%21%2168.52%2130.84%21%402101d3fe17896591844657076e1299%2112000046703163562%21rec%21MX%21895102358%21XZ%211%210%21n_tag%3A-29919%3Bd%3A9a638db6%3Bm03_new_user%3A-29895&utparam-url=scene%3ApcDetailTopMoreOtherSeller%7Cquery_from%3A%7Cx_object_id%3A1005008797499352%7C_p_origin_prod%3A",
            "https://es.aliexpress.com/item/1005012931736955.html?spm=a2g0o.productlist.main.48.7f1618d3NkUuQZ&aem_p4p_detail=202609162126501252617311745000000135374&algo_pvid=60d84d26-eabb-4746-8c97-b7cf0f36f73b&algo_exp_id=60d84d26-eabb-4746-8c97-b7cf0f36f73b-47&pdp_ext_f=%7B%22order%22%3A%22-1%22%2C%22eval%22%3A%221%22%2C%22fromPage%22%3A%22search%22%7D&pdp_npi=6%40dis%21MXN%21258.48%21118.90%21%21%2198.51%2145.31%21%40210311c217896192108697324e0e32%2112000059795146054%21sea%21MX%21895102358%21X%211%210%21n_tag%3A-29919%3Bd%3A9a638db6%3Bm03_new_user%3A-29895&curPageLogUid=1hW8I6wrOiJB&utparam-url=scene%3Asearch%7Cquery_from%3A%7Cx_object_id%3A1005012931736955%7C_p_origin_prod%3A&search_p4p_id=202609162126501252617311745000000135374_12",
            "https://es.aliexpress.com/item/1005008875165003.html?spm=a2g0o.productlist.main.4.7f1618d3NkUuQZ&aem_p4p_detail=202609162126501252617311745000000135374&algo_pvid=60d84d26-eabb-4746-8c97-b7cf0f36f73b&algo_exp_id=60d84d26-eabb-4746-8c97-b7cf0f36f73b-3&pdp_ext_f=%7B%22order%22%3A%22-1%22%2C%22spu_best_type%22%3A%22price%22%2C%22eval%22%3A%221%22%2C%22fromPage%22%3A%22search%22%7D&pdp_npi=6%40dis%21MXN%2177.56%2166.14%21%21%2129.56%2125.21%21%40210311c217896192108697324e0e32%2112000052881000092%21sea%21MX%21895102358%21X%211%210%21n_tag%3A-29919%3Bd%3A9a638db6%3Bm03_new_user%3A-29895%3BpisId%3A5000000217788164&curPageLogUid=H9mYS3qqyU2d&utparam-url=scene%3Asearch%7Cquery_from%3A%7Cx_object_id%3A1005008875165003%7C_p_origin_prod%3A&search_p4p_id=202609162126501252617311745000000135374_1",
            "https://es.aliexpress.com/item/1005011914108981.html?pdp_npi=4%40dis%21MXN%2195.62+MXN%24%2145.90+MXN%24%21%21%2136.44%2117.49%21%402101e75417896585370655866e0fe3%2112000057079121322%21sh%21MX%21895102358%21X&spm=a2g0o.store_pc_allItems_or_groupList.new_all_items_2009573860043.1005011914108981&gatewayAdapt=glo2esp",
            "https://es.aliexpress.com/item/1005010635630400.html?pdp_npi=4%40dis%21MXN%21116.20+MXN%24%21106.90+MXN%24%21%21%2144.28%2140.74%21%402101e75417896585370655866e0fe3%2112000053052906628%21sh%21MX%21895102358%21X&spm=a2g0o.store_pc_allItems_or_groupList.new_all_items_2009573860043.1005010635630400&gatewayAdapt=glo2esp",
            "https://es.aliexpress.com/item/1005011560839075.html?pdp_npi=4%40dis%21MXN%21241.46+MXN%24%21115.90+MXN%24%21%21%2192.02%2144.17%21%402101e75417896585370655866e0fe3%2112000055934851553%21sh%21MX%21895102358%21X&spm=a2g0o.store_pc_allItems_or_groupList.new_all_items_2009573860043.1005011560839075&gatewayAdapt=glo2esp",
            "https://es.aliexpress.com/item/1005011878729404.html?pdp_npi=4%40dis%21MXN%21114.37+MXN%24%2154.90+MXN%24%21%21%2143.59%2120.93%21%402101e75417896585370655866e0fe3%2112000056908548249%21sh%21MX%21895102358%21X&spm=a2g0o.store_pc_allItems_or_groupList.new_all_items_2009573860043.1005011878729404&gatewayAdapt=glo2esp"
        ]
    }
    run = RunManager.start_run(instance_id=EXAMPLE_INSTANCE_ID) 
    headless = False


In [5]:
run_state = run.refresh()
STEP = run_state.get('step', 'DATA_RECOLLECTION')
run_data = run_state.get('run_data', {})

## 2. Data Recollection

In [6]:
async def start_scraper():
    if STEP == 'DATA_RECOLLECTION':
        scraper = BaseScraper(
            headless=headless,
            use_proxy=True,
            proxy_user=DECODO_USERNAME,
            proxy_pass=DECODO_PASSWORD,
            proxy_server=DECODO_URL,
            remote_url=SELENIUM_REMOTE_URL,
            full_render=True
        )
        await scraper.start()
        return scraper
    return None

### Listing Page

#### Elements

In [7]:
LISTING_TITLE = "//div[contains(@class, 'title--wrap')]/h1"
ITEM_PROPERTIES = "//div[contains(@class, 'sku-item--property')]"
PROPERTIES_TEXT_BTN = "//div[contains(@class, 'sku-item--text')]"
PROPERTIES_IMAGE_BTN = "//div[contains(@class, 'sku-item--image')]"
PROPERTY_TITLE = "//div[contains(@class, 'sku-item--title')]"
PRPERTY_IMG = "//img[contains(@class, 'magnifier--image')]"
VIEW_MORE_PROPERTIES_BTN = "//div[contains(@class, 'sku-item--viewMore')]"
SPECIFICATIONS_LIST = "//ul[contains(@class, 'specification--list')]"
PRICE = " //span[contains(@class, 'price-default--current')]"
DESCRIPTION_WRAP = " //div[contains(@class, 'description--wrap')]"

#### Actions

In [8]:
await scraper.stop()

NameError: name 'scraper' is not defined

In [9]:
data_dict = {}
listings_urls = PARAMS['listings_urls']
drive = DriveApi()

# Navigate to the target product page and simulate human delay to avoid bot detection.
listing = listings_urls[4]

is_clean_proxy = False
proxy_count = 0
while not is_clean_proxy and proxy_count < 6:
    proxy_count = proxy_count + 1
    scraper = await start_scraper()
    await scraper.page.goto(listing, timeout=60000)
    await scraper.wait_for_page_load()
    current_url = scraper.page.url
    if 'punish' not in current_url:
        is_clean_proxy = True

try:
    await scraper.human_pause()  # Wait 1 to 3 seconds
    await scraper.page.reload()
    # Pauses for exactly 3 seconds (3000 milliseconds)
    # await scraper.page.wait_for_timeout(3000)
    # await scraper.wait_for_page_load()
    # await scraper.human_pause()  # Wait 1 to 3 seconds
    # await scraper.page.goto(listing)
    # await scraper.wait_for_page_load()
except TimeoutError:
    raise TimeoutError


await scraper.human_pause() 
await scraper.human_pause() 
# --- PHASE 3: CORE LISTING METADATA EXTRACTION ---
# Attempt to extract the primary title. If this fails, the page likely didn't render correctly,
# so we skip this iteration entirely to prevent cascading errors.
try:
    listing_title = await scraper.page.locator(LISTING_TITLE).inner_text(timeout=5000)
except TimeoutError:
    print(f"Failed to load listing title for {listing}. Skipping to next URL.")
    pass 

data_dict[listing_title] = {} 

# Extract HTML specifications list and plain text description.
# Using try-except blocks ensures that missing optional blocks do not crash the pipeline.
try:
    specifications = await scraper.page.locator(SPECIFICATIONS_LIST).inner_html(timeout=3000)
except TimeoutError:
    specifications = ""
data_dict[listing_title]['specifications'] = specifications

try:
    description = await scraper.page.locator(DESCRIPTION_WRAP).inner_text(timeout=3000)
except TimeoutError:
    description = ""
data_dict[listing_title]['description'] = description

# --- PHASE 4: TEXT-BASED VARIANTS EXTRACTION ---
# Look for variants that are text-only (e.g., sizes or simple configuration options).
item_text_property_locator = scraper.page.locator(PROPERTIES_TEXT_BTN)
try:
    await item_text_property_locator.first.wait_for(timeout=3000)
    item_text_properties = await item_text_property_locator.all_inner_texts()
    data_dict[listing_title]['text_properties'] = item_text_properties
except TimeoutError:
    pass

# --- PHASE 5: EXPANDING IMAGE VARIANTS ---
# If the product has many visual variants, they might be hidden behind a "View More" button.
# We must reveal all of them before we can iterate through the thumbnails.
view_more_properties_btn_locator = scraper.page.locator(VIEW_MORE_PROPERTIES_BTN)
try:
    await view_more_properties_btn_locator.first.wait_for(timeout=3000)
    await scraper.human_pause()
    await view_more_properties_btn_locator.click()
except TimeoutError:
    pass

# --- PHASE 6: ITERATING OVER IMAGE VARIANTS ---
data_dict[listing_title]['img_properties'] = {}
item_img_properties = await scraper.page.locator(PROPERTIES_IMAGE_BTN).all()

for i, img_property in enumerate(item_img_properties):
    # Click the variant thumbnail to trigger DOM updates for the specific variant's price and large image.
    await scraper.human_jitter_click(img_property)
    await scraper.human_pause()

    # --- PHASE 7: VARIANT SPECIFIC DATA EXTRACTION ---
    try:
        property_title = await scraper.page.locator(PROPERTY_TITLE).inner_text(timeout=3000)
    except TimeoutError:
        property_title = f"Unknown_Variant_{i}"

    try:
        # Extract the high-resolution image URL.
        img_url = await scraper.page.locator(PRPERTY_IMG).get_attribute("src", timeout=3000)
        if img_url and img_url.startswith("//"):
            img_url = "https:" + img_url

        # --- PHASE 8: IN-MEMORY DOWNLOADING AND CLOUD UPLOAD ---
        # Download the image directly to memory as bytes via Playwright's network router.
        response = await scraper.page.request.get(img_url)
        image_bytes = await response.body()
        raw_filename = img_url.split('/')[-1].split('?')[0]

        # Sanitize the listing title to ensure it doesn't create unwanted subdirectories in Google Drive.
        safe_title = listing_title.replace('/', '-').replace('\\', '-').replace(':', '')
        
        # Offload synchronous Google Drive API calls to a separate thread to keep the async loop unblocked.
        drive_path = f'LabScrapersPipeline/WatchParts/{safe_title}'
        drive_folder_id = drive.get_or_create_path(drive_path)

        file_id = drive.upload_file(
            file_input=image_bytes,
            folder_id=drive_folder_id,
            filename=raw_filename,
            mimetype="image/avif" if ".avif" in raw_filename else "image/jpeg"
        )
    except Exception as e:
        print(f"Image extraction/upload failed for {property_title}: {e}")
        file_id = None

    # Extract the dynamic price associated with this specific variant click.
    try:
        property_price = await scraper.page.locator(PRICE).inner_text(timeout=3000)
    except TimeoutError:
        property_price = "Price not found"
    
    # --- PHASE 9: VARIANT DATA AGGREGATION ---
    data_dict[listing_title]['img_properties'][property_title] = {
        "drive_file_id": file_id,
        "price": property_price
    }

print(data_dict)

Configuring Decodo proxy: Targeting Mexico (MX) Residential Node...
Connecting to remote VNC container via CDP: ws://chrome:3000
Scraper successfully started with data-saving routing rules.


TimeoutError: Page.goto: Timeout 60000ms exceeded.
Call log:
  - navigating to "https://es.aliexpress.com/item/1005012308067099.html?spm=a2g0o.store_pc_allItems_or_groupList.0.0.43045a997SibTa&pdp_npi=4%40dis%21MXN%211%2C246.86+MXN%24%21635.90+MXN%24%21%21%2170.80%2136.11%21%4021030cd817896187107086825e0e25%2112000058095122510%21sh%21MX%21895102358%21X&_gl=1*1aqi1s4*_gcl_aw*R0NMLjE3ODk2MTgzODUuQ2p3S0NBandfS2pWQmhBSEVpd0FuQzBOOUhld3BXcHB3VlRqUW1OZkFseVd3b3VyM3NJb1NhM1VURGwySGxBNjhKZGdHeUk2MDUzNE9Sb0N2b2NRQXZEX0J3RQ..*_gcl_dc*R0NMLjE3ODk2MTgzODUuQ2p3S0NBandfS2pWQmhBSEVpd0FuQzBOOUhld3BXcHB3VlRqUW1OZkFseVd3b3VyM3NJb1NhM1VURGwySGxBNjhKZGdHeUk2MDUzNE9Sb0N2b2NRQXZEX0J3RQ..*_gcl_au*NDAwMTA4MjYzLjE3ODQ2NDgxNjc.*_ga*ODAxMTU1NzM2LjE3NzY3OTgxOTU.*_ga_VED1YSGNC7*czE3ODk2MTgzNjEkbzgkZzEkdDE3ODk2MTg3MDgkajUkbDAkaDA.&gatewayAdapt=glo2esp", waiting until "load"


In [ ]:
try:
    if STEP == 'DATA_RECOLLECTION':
        # --- PHASE 1: PIPELINE INITIALIZATION ---
        # Initialize the main data payload and instantiate the Google Drive API client.
        data_dict = {}
        listings_urls = PARAMS['listings_urls']
        drive = DriveApi()

        # --- PHASE 2: ITERATING OVER TARGET URLS ---
        for listing in listings_urls:
            # Navigate to the target product page and simulate human delay to avoid bot detection.
            await scraper.page.goto(listing)
            await scraper.human_pause()

            # --- PHASE 3: CORE LISTING METADATA EXTRACTION ---
            # Attempt to extract the primary title. If this fails, the page likely didn't render correctly,
            # so we skip this iteration entirely to prevent cascading errors.
            try:
                listing_title = await scraper.page.locator(LISTING_TITLE).inner_text(timeout=5000)
            except TimeoutError:
                print(f"Failed to load listing title for {listing}. Skipping to next URL.")
                pass 
            
            data_dict[listing_title] = {} 

            # Extract HTML specifications list and plain text description.
            # Using try-except blocks ensures that missing optional blocks do not crash the pipeline.
            try:
                specifications = await scraper.page.locator(SPECIFICATIONS_LIST).inner_html(timeout=3000)
            except TimeoutError:
                specifications = ""
            data_dict[listing_title]['specifications'] = specifications

            try:
                description = await scraper.page.locator(DESCRIPTION_WRAP).inner_text(timeout=3000)
            except TimeoutError:
                description = ""
            data_dict[listing_title]['description'] = description

            # --- PHASE 4: TEXT-BASED VARIANTS EXTRACTION ---
            # Look for variants that are text-only (e.g., sizes or simple configuration options).
            item_text_property_locator = scraper.page.locator(PROPERTIES_TEXT_BTN)
            try:
                await item_text_property_locator.first.wait_for(timeout=3000)
                item_text_properties = await item_text_property_locator.all_inner_texts()
                data_dict[listing_title]['text_properties'] = item_text_properties
            except TimeoutError:
                pass
            
            # --- PHASE 5: EXPANDING IMAGE VARIANTS ---
            # If the product has many visual variants, they might be hidden behind a "View More" button.
            # We must reveal all of them before we can iterate through the thumbnails.
            view_more_properties_btn_locator = scraper.page.locator(VIEW_MORE_PROPERTIES_BTN)
            try:
                await view_more_properties_btn_locator.first.wait_for(timeout=3000)
                await scraper.human_pause()
                await scraper.human_jitter_click(view_more_properties_btn_locator)
            except TimeoutError:
                pass

            # --- PHASE 6: ITERATING OVER IMAGE VARIANTS ---
            data_dict[listing_title]['img_properties'] = {}
            item_img_properties = await scraper.page.locator(PROPERTIES_IMAGE_BTN).all()
            
            for i, img_property in enumerate(item_img_properties):
                # Click the variant thumbnail to trigger DOM updates for the specific variant's price and large image.
                await scraper.human_jitter_click(img_property)
                await scraper.human_pause()

                # --- PHASE 7: VARIANT SPECIFIC DATA EXTRACTION ---
                try:
                    property_title = await scraper.page.locator(PROPERTY_TITLE).inner_text(timeout=3000)
                except TimeoutError:
                    property_title = f"Unknown_Variant_{i}"

                try:
                    # Extract the high-resolution image URL.
                    img_url = await scraper.page.locator(PRPERTY_IMG).get_attribute("src", timeout=3000)
                    if img_url and img_url.startswith("//"):
                        img_url = "https:" + img_url

                    # --- PHASE 8: IN-MEMORY DOWNLOADING AND CLOUD UPLOAD ---
                    # Download the image directly to memory as bytes via Playwright's network router.
                    response = await scraper.page.request.get(img_url)
                    image_bytes = await response.body()
                    raw_filename = img_url.split('/')[-1].split('?')[0]

                    # Sanitize the listing title to ensure it doesn't create unwanted subdirectories in Google Drive.
                    safe_title = listing_title.replace('/', '-').replace('\\', '-').replace(':', '')
                    
                    # Offload synchronous Google Drive API calls to a separate thread to keep the async loop unblocked.
                    drive_path = f'LabScrapersPipeline/WatchParts/{safe_title}'
                    drive_folder_id = await asyncio.to_thread(drive.get_or_create_path, drive_path)

                    file_id = await asyncio.to_thread(
                        drive.upload_file, 
                        file_input=image_bytes,
                        folder_id=drive_folder_id,
                        filename=raw_filename,
                        mimetype="image/avif" if ".avif" in raw_filename else "image/jpeg"
                    )
                except Exception as e:
                    print(f"Image extraction/upload failed for {property_title}: {e}")
                    file_id = None

                # Extract the dynamic price associated with this specific variant click.
                try:
                    property_price = await scraper.page.locator(PRICE).inner_text(timeout=3000)
                except TimeoutError:
                    property_price = "Price not found"
                
                # --- PHASE 9: VARIANT DATA AGGREGATION ---
                data_dict[listing_title]['img_properties'][property_title] = {
                    "drive_file_id": file_id,
                    "price": property_price
                }
                
    elif STEP in ['STORING_RAW_DATA', 'CLEANING_RAW_DATA']:
        # Retrieve the accumulated data payload for downstream pipeline steps.
        data_dict = run.get_raw_data()
        
finally:
    # --- PHASE 10: PIPELINE TEARDOWN ---
    # Ensure browser instances and background processes are gracefully terminated to prevent memory leaks.
    if STEP == 'DATA_RECOLLECTION' and 'scraper' in locals():
        await scraper.stop()

Scraper safely closed.


TimeoutError: Page.goto: Timeout 30000ms exceeded.
Call log:
  - navigating to "https://es.aliexpress.com/item/1005009141784587.html?spm=a2g0o.store_pc_home.promoteWysiwyg_2010150883161.1005009141784587&gatewayAdapt=glo2esp", waiting until "load"


### Storing Raw Data

In [ ]:
if STEP == 'DATA_RECOLLECTION':
    STEP = 'STORING_RAW_DATA'
    run.update(step = STEP)

In [ ]:
if STEP == 'STORING_RAW_DATA':
    try:
        if not data_dict:
            raise ValueError("Zero items collected.")
        
        if run: run.save_raw_data(data_dict)
    except Exception as e:
        handle_error(PIPELINE_NAME, run, "STORING_RAW_DATA", e)

## 4. Data Cleaning

In [ ]:
if STEP == 'STORING_RAW_DATA':
    STEP = 'CLEANING_RAW_DATA'
    run.update(step = STEP)

In [ ]:
if STEP == 'CLEANING_RAW_DATA':
    try:
        df = pl.DataFrame(data_dict)
        df = strip_all_str(df)
        
        # Standard cleaning/casting
        df = df.with_columns([
            pl.col("Team Name").fill_null("Unknown").cast(pl.String),
            pl.col("Year").replace("","0").cast(pl.Int64),
            pl.col("Wins").replace("","0").cast(pl.Int64),
            pl.col("Losses").replace("","0").cast(pl.Int64),
            pl.col("OT Losses").replace("","0").cast(pl.Int64),
            pl.col("Win %").cast(pl.Float64),
            pl.col("Goals For (GF)").replace("","0").cast(pl.Int64),
            pl.col("Goals Against (GA)").replace("","0").cast(pl.Int64),
            pl.col("+ / -").replace("","0").cast(pl.Int64)
        ])
        
        cleaned_data = df.to_dict(as_series=False)
    except Exception as e:
        handle_error(PIPELINE_NAME, run, "CLEANING_RAW_DATA", e)

### Storing Cleaned Data

In [ ]:
if STEP == 'CLEANING_RAW_DATA':
    STEP = 'STORING_CLEANED_DATA'
    run.update(step = STEP)

In [ ]:
if STEP == 'STORING_CLEANED_DATA':
    try:
        if run: run.save_cleaned_data(cleaned_data)
        print("Done.")
    except Exception as e:
        handle_error(PIPELINE_NAME, run, "STORING_CLEANED_DATA", e)

Done.
